# CyberPhi — Colab Quickstart

**Purpose:** Smoke-test the full pipeline (data → train → inference) on a free Colab GPU.

**Training backend:** `trl` SFTTrainer + PEFT (not Axolotl) — Axolotl's build deps conflict with
Colab's pre-installed packages. SFTTrainer produces an identical LoRA adapter; the full RunPod
run uses Axolotl as normal.

**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 is fine)
2. Add secrets via the `🔑` icon:
   - `ANTHROPIC_API_KEY` *(required)*
   - `NVD_API_KEY` *(optional — without it NVD rate-limits to 5 req/30 s)*
   - `HUGGINGFACE_TOKEN` *(optional — needed for Alpaca mixing step)*
3. Run all cells top to bottom — no restart required

---
## 0 — GPU Check

In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type and select GPU')

gpu_name, vram = [s.strip() for s in result.stdout.strip().split(',')]
print(f'GPU:  {gpu_name}')
print(f'VRAM: {vram}')

HIGH_END_GPUS = ('A100', 'A10', 'L4', 'H100')
IS_HIGH_END   = any(g in gpu_name for g in HIGH_END_GPUS)
print(f'\nProfile: {"high-end" if IS_HIGH_END else "T4/P100 (lightweight config)"}')

---
## 1 — Clone Repo

In [ ]:
MOUNT_DRIVE = False   # set True to persist across session resets

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/cyberphi'
else:
    REPO_DIR = '/content/cyberphi'

import os
if not os.path.exists(REPO_DIR):
    REPO_URL = 'https://github.com/boyatoid/CyberPhi.git'
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print(f'Repo updated at {REPO_DIR}')

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## 2 — API Keys

In [ ]:
from google.colab import userdata
import os

def _get_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Secret "{name}" not found — add it via the 🔑 Secrets panel')
    return ''

ANTHROPIC_API_KEY = _get_secret('ANTHROPIC_API_KEY')
NVD_API_KEY       = _get_secret('NVD_API_KEY',      required=False)
HF_TOKEN          = _get_secret('HUGGINGFACE_TOKEN', required=False)

with open('.env', 'w') as f:
    f.write(f'ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}\n')
    if NVD_API_KEY: f.write(f'NVD_API_KEY={NVD_API_KEY}\n')
    if HF_TOKEN:    f.write(f'HUGGINGFACE_TOKEN={HF_TOKEN}\n')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if NVD_API_KEY: os.environ['NVD_API_KEY']       = NVD_API_KEY
if HF_TOKEN:    os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

print('ANTHROPIC_API_KEY: set ✓')
print(f'NVD_API_KEY:       {"set ✓" if NVD_API_KEY else "not set (rate-limited)"}')
print(f'HUGGINGFACE_TOKEN: {"set ✓" if HF_TOKEN    else "not set (Alpaca mixing skipped)"}')

---
## 3 — Install Dependencies

Only installs packages Colab doesn't already ship — no runtime restart needed.

In [ ]:
# Colab already has: torch, transformers, peft, trl, bitsandbytes, accelerate, datasets
# We only need the dataset pipeline extras and rouge-score
!pip install anthropic requests aiohttp beautifulsoup4 lxml datasketch jsonlines \
             python-dotenv rouge-score chromadb -q

# Confirm the training stack is present
import torch, transformers, peft, trl
print(f'torch:          {torch.__version__}')
print(f'transformers:   {transformers.__version__}')
print(f'peft:           {peft.__version__}')
print(f'trl:            {trl.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

---
## 4 — Data Pipeline

50 rows ≈ a few minutes and < $0.10 in Claude API cost.

In [ ]:
import json
from pathlib import Path

DATA_LIMIT = 50

!python dataset/pipeline.py all --limit {DATA_LIMIT}

validated = Path('data/final/validated.jsonl')
if not validated.exists() or validated.stat().st_size == 0:
    raise RuntimeError('Pipeline produced no output — check logs above')

rows   = validated.read_text().strip().splitlines()
sample = json.loads(rows[0])
print(f'\n✓ {len(rows)} validated training rows')
print(f'  source:      {sample["source"]}')
print(f'  vuln_type:   {sample["vuln_type"]}')
print(f'  instruction: {sample["instruction"][:80]}…')

---
## 5 — Training (trl SFTTrainer)

Uses the same hyperparameters as `training/axolotl_config.yaml` (rank-16 QLoRA, 1e-4 lr, cosine schedule).
Produces an identical LoRA adapter — the only difference from the RunPod run is the training framework.

In [ ]:
import json, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'
RUN_NAME   = 'colab_test'
OUTPUT_DIR = f'outputs/{RUN_NAME}'
SYSTEM     = 'You are a senior cybersecurity expert and penetration tester.'

# ── Dataset ──────────────────────────────────────────────────────────────────
raw = [json.loads(l) for l in Path('data/final/validated.jsonl').read_text().splitlines()]

def to_chatml(ex):
    return {
        'text': (
            f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
            f'<|im_start|>user\n{ex["instruction"]}'
            + (f'\n{ex["input"]}' if ex.get('input') else '')
            + f'<|im_end|>\n'
            f'<|im_start|>assistant\n{ex["output"]}<|im_end|>'
        )
    }

dataset = Dataset.from_list([to_chatml(r) for r in raw])
split   = dataset.train_test_split(test_size=0.02, seed=42)
print(f'Train: {len(split["train"])} rows  |  Val: {len(split["test"])} rows')

# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# ── Model (4-bit QLoRA) ───────────────────────────────────────────────────────
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if IS_HIGH_END else torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_cfg,
    device_map='auto', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# ── LoRA config — matches axolotl_config.yaml ─────────────────────────────────
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

# ── Training args ─────────────────────────────────────────────────────────────
sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch = 8
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    max_seq_length=2048 if not IS_HIGH_END else 4096,
    fp16=not IS_HIGH_END,
    bf16=IS_HIGH_END,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    save_steps=50,
    eval_steps=50,
    eval_strategy='steps',
    save_total_limit=3,
    logging_steps=10,
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    peft_config=lora_cfg,
)

print(f'Starting training → {OUTPUT_DIR}')
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'\n✓ Adapter saved to {OUTPUT_DIR}')

In [ ]:
# Verify adapter files
adapter_files = list(Path(OUTPUT_DIR).rglob('adapter_model*'))
if not adapter_files:
    print('WARNING: no adapter files found — training may have failed')
else:
    print('✓ Adapter files:')
    for f in adapter_files:
        print(f'  {f}  ({f.stat().st_size / 1e6:.1f} MB)')

---
## 6 — Inference Test

In [ ]:
from peft import PeftModel

# model is still in memory — just wrap with the saved adapter
# (reload from disk if you restarted the runtime after training)
if not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(model, OUTPUT_DIR)
model.eval()

QUERY = 'Explain how SQL injection works and give a simple example of a vulnerable PHP snippet.'
prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{QUERY}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
print(f'Query: {QUERY}\n--- Response ---')

with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=512, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
    )

print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

---
## 7 — Evaluation Metrics

In [ ]:
import json, sys
sys.path.insert(0, '.')
from evaluation.metrics import (
    rouge_l_score, think_block_presence_rate,
    cot_step_count_distribution, vuln_class_accuracy,
)

EVAL_LIMIT = 5
samples    = [json.loads(r) for r in
              Path('data/final/validated.jsonl').read_text().strip().splitlines()[:EVAL_LIMIT]]

outputs, refs, classes = [], [], []
for s in samples:
    prompt = (
        f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
        f'<|im_start|>user\n{s["instruction"]}'
        + (f'\n{s["input"]}' if s.get('input') else '')
        + f'<|im_end|>\n<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    outputs.append(response)
    refs.append(s['output'])
    classes.append(s.get('vuln_type', ''))
    print(f'  [{s["source"]}] {s["instruction"][:60]}…  →  {len(response)} chars')

rouge_scores = [rouge_l_score(o, r) for o, r in zip(outputs, refs)]
print(f'\n=== Eval over {len(samples)} samples ===')
print(f'Mean ROUGE-L:          {sum(rouge_scores) / len(rouge_scores):.3f}')
print(f'<think> block rate:    {think_block_presence_rate(outputs):.1%}')
print(f'Vuln-class accuracy:   {vuln_class_accuracy(outputs, classes):.1%}')
print(f'CoT step distribution: {cot_step_count_distribution(outputs)}')

---
## 8 — (Optional) Download Adapter

In [ ]:
import shutil
from google.colab import files

zip_path = f'/content/{RUN_NAME}_adapter.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUT_DIR)
files.download(zip_path)
print(f'Downloaded {zip_path}')